# Kimi K3 · Generation playground

A reusable interactive notebook for **any compatible trainer-format (v7) checkpoint**. It uses `src/kimi_k3/inference.py` and `src/kimi_k3/evaluation.py` rather than the frozen M9 scripts.

This notebook is exploratory. It does **not** reproduce M9.3's held-out 64-window evaluation or M9.4's pre-registered generation battery. See the `pre-cleanup-m9` Git tag for the original runnable M9 code.

Requirements: project environment with PyTorch, Transformers and the local checkpoint. Run from the repository root or from `notebooks/`. A CPU check is possible with the *tiny* model, but a ~300M checkpoint is much slower on CPU. The default KDA inference backend is the readable PyTorch `chunkwise` implementation; results must not be claimed numerically identical to fused training without a parity test.

## 1 · Load imports and identify the repository

In [ ]:
from pathlib import Path
import json, sys, time
import torch
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / 'src/kimi_k3/inference.py').is_file()), None)
if root is None:
    raise RuntimeError('Launch this notebook from within kimi-k3-from-scratch')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from kimi_k3.inference import load_checkpoint_model, resolve_device, generate_tokens
from kimi_k3.evaluation import evaluate_windows
from kimi_k3.data.dataset import ShardedTokenDataset
print('Repository:', root)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2 · Configure and load the model

Edit `CHECKPOINT` to point to a **trusted local** `.pt` checkpoint saved by this repository's trainer. Only model weights and configuration are restored (not optimizer or training RNG). `chunkwise` is the reference backend and works on CPU and CUDA. `fused` requires compatible CUDA + FLA. We default to FP32 for interpretability.

In [ ]:
CHECKPOINT = root / 'results/training/runs/REPLACE_WITH_YOUR_RUN/last.pt'
DEVICE = 'auto'            # 'auto', 'cuda', or 'cpu'
BACKEND = 'chunkwise'      # or 'fused' on compatible CUDA
PRECISION = 'fp32'        # or 'bf16' on compatible CUDA

checkpoint_path = Path(CHECKPOINT)
if not checkpoint_path.is_file():
    raise FileNotFoundError(f'Edit CHECKPOINT to a local trainer-format v7 .pt file: {checkpoint_path}')
device = resolve_device(DEVICE)
t0 = time.perf_counter()
model, metadata = load_checkpoint_model(checkpoint_path, device=device, backend=BACKEND)
print(json.dumps(metadata, indent=2))
print(f'Loaded to {device} in {time.perf_counter()-t0:.1f}s')

## 3 · Generate from a text prompt

The training tokenizer was the SmolLM2-135M tokenizer. Keep its revision pinned for meaningful comparisons; another tokenizer with the same vocabulary size is not automatically compatible. If you have cached the tokenizer, set `LOCAL_FILES_ONLY=True`.

Generation currently recomputes a forward pass at each token; there is **no KV/recurrent-state cache or sliding window**. To preserve causality, chunkwise KDA right-pads temporary chunks but only scores the last genuine prompt position.

In [ ]:
from transformers import AutoTokenizer
TOKENIZER = 'HuggingFaceTB/SmolLM2-135M'
TOKENIZER_REVISION = '93efa2f097d58c2a74874c7e644dbc9b0cee75a2'
LOCAL_FILES_ONLY = False

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER, revision=TOKENIZER_REVISION,
                                          use_fast=True, local_files_only=LOCAL_FILES_ONLY)
assert len(tokenizer) == model.config.vocab_size, 'Tokenizer vocabulary mismatch'
EOS_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0
print('Tokenizer:', TOKENIZER, '| EOS:', EOS_ID)

In [ ]:
PROMPT = 'The results of the experiment showed that'
MODE = 'greedy'           # 'greedy' or 'sample'
MAX_NEW_TOKENS = 32
TEMPERATURE = 1.0
TOP_K = 50
TOP_P = 0.95
SEED = 1337

prompt_ids = tokenizer.encode(PROMPT, add_special_tokens=False)
if not prompt_ids:
    raise ValueError('Empty tokenized prompt')
if len(prompt_ids) + MAX_NEW_TOKENS > model.config.max_seq_len:
    raise ValueError('Prompt plus generated-token budget exceeds max_seq_len')

t0 = time.perf_counter()
new_ids = generate_tokens(model, prompt_ids, device=device,
                          max_new_tokens=MAX_NEW_TOKENS, mode=MODE,
                          temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
                          seed=SEED, eos_id=EOS_ID, precision=PRECISION)
completion = tokenizer.decode(new_ids, skip_special_tokens=True)
display(Markdown('**Prompt:** ' + PROMPT + '\n\n**Completion:** ' + completion))
print('Generated tokens:', len(new_ids), '| Seconds:', round(time.perf_counter()-t0, 2))

## 4 · Compare decoding strategies (optional)

Greedy decoding is deterministic for a fixed model/backend/precision. With sampling, a fixed seed makes the token draws repeatable for the same logits, but a seed does not guarantee identical results across different numerical backends.

In [ ]:
RUN_COMPARISON = False
if RUN_COMPARISON:
    for label, mode, temperature in [('greedy','greedy',1.0),
                                      ('sample_0.7','sample',0.7),
                                      ('sample_1.0','sample',1.0),
                                      ('sample_1.3','sample',1.3)]:
        ids = generate_tokens(model, prompt_ids, device=device,
                              max_new_tokens=MAX_NEW_TOKENS, mode=mode,
                              temperature=temperature, top_k=TOP_K, top_p=TOP_P,
                              seed=SEED, eos_id=EOS_ID, precision=PRECISION)
        print(f'{label}: {tokenizer.decode(ids, skip_special_tokens=True)}\n')

## 5 · Exploratory next-token evaluation (optional)

This scores **consecutive non-overlapping windows** of the prepared validation stream. Configure a prepared dataset with `metadata.json` and `val_*.npy` shards. It does not exclude the historical M9 monitored prefix or use M9's registered offsets. **Do not compare the resulting perplexity directly to the published M9.3 result** without matching all evaluation conditions.

In [ ]:
RUN_EVALUATION = False
DATA_DIR = Path('C:/datasets/fineweb_edu_6b')  # edit to your own prepared dataset
EVAL_SEQ_LEN = min(128, model.config.max_seq_len)
EVAL_WINDOWS = 4
EVAL_START_OFFSET = 0

if RUN_EVALUATION:
    ds = ShardedTokenDataset(DATA_DIR, split='val')
    if model.config.kda_implementation == 'chunkwise' and 'kda' in model.config.layer_types:
        EVAL_SEQ_LEN = (EVAL_SEQ_LEN // model.config.kda_chunk_size) * model.config.kda_chunk_size
        if not EVAL_SEQ_LEN:
            raise ValueError('Context too small for one chunk')
    summary = evaluate_windows(model, ds, device=device, seq_len=EVAL_SEQ_LEN,
                               num_windows=EVAL_WINDOWS, start_offset=EVAL_START_OFFSET,
                               precision=PRECISION)
    print(json.dumps(summary, indent=2))

## 6 · Interpretation

- A plausible completion is **not** evidence of generalization.
- Top-1 accuracy and perplexity are meaningful only with stated data, tokenizer, backend and scoring policy.
- The frozen M9 qualitative and quantitative protocols are historical scientific artifacts, not defaults of this playground.
- When sharing results, record the checkpoint hash, model revision, dataset manifest hash, tokenizer revision, precision and inference backend.